# Certification — `STL_Healpix_Kernel_Torch`: interface parity with the 2D kernel

This notebook certifies **steps 1 to 4** of the HEALPix port:

1. `Base_DataClass` and `ST_Operator` made data-type agnostic
   (`NDIM_PIX`, `_infer_N0`, no hard-coded pixel axes in the slicing);
2. `STL_Healpix_Kernel_Torch` derived from `Base_DataClass`
   (`pbc`, `dg`, `N0`, `conv_history`, `cell_ids`, `divide`, `get_ST_op`);
3. the statistics moved onto the wavelet operator
   (`mean`, `square_mean`, `cov`, `standardize`, `unstandardize`,
   `_compute_and_store_cross_cov`, `j_to_dg`, `mask_full_res`, `downsample`);
4. `get_CS_op()` — the angular power spectrum, built on the differentiable
   spherical harmonic transform of `healpix-analyse`
   (`HEALPixSHT`: `map2alm` / `alm2map` / `anafast`).

**Out of scope at this stage** (see the closing section):

* the full mask / NaN bookkeeping (`mask_full_res`), whose spherical
  counterpart is the erosion of the validity mask by the stencil support;
* the deconvolution of the mask-induced mode coupling: the partial-sky
  spectrum is a pseudo-C_ell, with no MASTER matrix.

In [1]:
import os
import sys
import warnings

import numpy as np
import torch
import healpy as hp
import matplotlib.pyplot as plt

# --- walk up to the repository root (the directory holding STL_main) ---
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "STL_main")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("STL_main directory not found above %s" % os.getcwd())
    ROOT = parent
sys.path.insert(0, ROOT)

DATA_TEST_PATH = os.path.join(ROOT, "data", "test")
print("Repository root:", ROOT)

warnings.filterwarnings("ignore", category=UserWarning)

from STL_main.ST_Operator import ST_Operator
from STL_main.STL_Healpix_Kernel_Torch import (
    STL_Healpix_Kernel_Torch as DataClass,
    WaveletOperatorHealpixKernel_torch,
)

torch.manual_seed(0)
np.random.seed(0)

Repository root: /home/claude/work/STL-Dev


## 0. Test map

The repository LSS map is used when present, otherwise a Gaussian realisation of
a power-law spectrum. **NESTED** ordering in both cases, as the HEALPix
downsampling requires it.

In [2]:
NSIDE = 32
NPIX = 12 * NSIDE**2

lss_file = os.path.join(DATA_TEST_PATH, "Test_Heal_LSS.npy")
if os.path.exists(lss_file):
    heal_im = np.load(lss_file)
    heal_im = np.mean(heal_im.reshape(NPIX, heal_im.shape[0] // NPIX), 1)
    origin = "Test_Heal_LSS.npy"
else:
    ell = np.arange(3 * NSIDE)
    cl = 1.0 / (ell + 10.0) ** 2.5
    heal_im = hp.reorder(hp.synfast(cl, NSIDE), r2n=True)
    origin = "synfast (power law)"

heal_im = (heal_im - heal_im.mean()) / heal_im.std()
print("map:", origin, "| nside =", NSIDE, "| Npix =", heal_im.shape)

hp.mollview(heal_im, nest=True, cmap="plasma", title="Test map (NESTED)")
plt.show()

map: synfast (power law) | nside = 32 | Npix = (12288,)


## 1. The data class does derive from `Base_DataClass`

Every attribute the data-type agnostic code relies on must be there: `DT`,
`NDIM_PIX`, `N0`, `dg`, `pbc`, `conv_history`, plus the HEALPix-specific
`cell_ids` / `nest` / `nside`.

### Dependencies

The HEALPix kernel relies on **`healpix-analyse`** for the spherical geometry
(convolution and resolution changes). The reference is the *local head* of the
sibling repository, installed in editable mode from the checkout:

```bash
pip install -e ../healpix-analyse
```

The signature of `HealPixConv` changed across releases (the resolution argument
was named `level`, then `nside`); the kernel adapts to whichever is installed,
and the cell below reports the one actually in use. No other scattering package
is required.

In [3]:
import inspect

import healpix_analyse
from healpix_analyse.convol import HealPixConv
from healpix_analyse.down import HealPixDown

conv_params = list(inspect.signature(HealPixConv.__init__).parameters)
resolution_arg = next(a for a in ("nside", "level") if a in conv_params)

print("healpix-analyse                 :", os.path.dirname(healpix_analyse.__file__))
print("HealPixConv resolution argument :", resolution_arg)

assert "foscat" not in sys.modules, "the kernel must not depend on any other scattering package"
print("no other scattering package imported")

from STL_main.torch_backend import _DEFAULT_DEVICE, _DEFAULT_DTYPE

print("default device / dtype          :", _DEFAULT_DEVICE, "/", _DEFAULT_DTYPE)

healpix-analyse                 : /home/claude/hpa_dev/healpix_analyse
HealPixConv resolution argument : level
no other scattering package imported
default device / dtype          : cpu / torch.float64


In [4]:
from STL_main.Base_DataClass import Base_DataClass

data = DataClass(heal_im)

assert isinstance(data, Base_DataClass), "the class must derive from Base_DataClass"
assert data.NDIM_PIX == 1
assert data.N0 == (NSIDE,), data.N0
assert len(data.N0) == data.NDIM_PIX, "len(N0) must equal NDIM_PIX"
assert data.dg == 0 and data.nside == NSIDE
assert data.pbc is True, "full sky -> pbc True"
assert data.conv_history == []
assert data.cell_ids.shape == (NPIX,)

print("DT            :", data.DT)
print("N0 / dg       :", data.N0, "/", data.dg)
print("current nside :", data.nside)
print("pbc           :", data.pbc)
print("device/dtype  :", data.device, data.dtype)
print("cell_ids      :", data.cell_ids[:5].tolist(), "...")

DT            : HealpixKernel_torch
N0 / dg       : (32,) / 0
current nside : 32
pbc           : True
device/dtype  : cpu torch.float64
cell_ids      : [0, 1, 2, 3, 4] ...


### `copy`, `__getitem__`, `modulus`, `divide`

`divide` is what the cross-channel S1 terms need (`I*psi / |I*psi|^0.5`); it was
missing entirely from the previous version.

In [5]:
batch = DataClass(np.stack([heal_im, -heal_im]))       # (2, Npix)

sub = batch[0]                                          # slicing over leading dims
assert sub.array.shape == (NPIX,)
assert sub.cell_ids.shape == (NPIX,)

cp = batch.copy()
cp.array[:] = 0.0
assert not torch.allclose(cp.array, batch.array), "copy() must be deep on array"

mod = batch.modulus()
assert torch.allclose(mod.array, batch.array.abs())

num = DataClass(np.ones((2, NPIX)))
den = DataClass(4.0 * np.ones((2, NPIX)))
div = num.divide(den, epsilon=0.0, pow=0.5)
assert torch.allclose(div.array, torch.full_like(div.array, 0.5))

print("copy / __getitem__ / modulus / divide: OK")

copy / __getitem__ / modulus / divide: OK


## 2. Wavelet operator: spherical convolution

The convolution is the one from **`healpix-analyse`** (`HealPixConv`,
gauge-equivariant): the complex kernel is carried as two output channels (real
and imaginary parts) and the `L` orientations are the `L` gauges, so a single
call produces the complex answer.

`apply(data, j)` returns `[..., L, Npix]` and now fills `conv_history`, on which
the per-layer mask selection depends.

In [6]:
J, L, KERNELSZ = 4, 4, 5
wav_op = data.get_wavelet_op(J=J, L=L, kernel_size=KERNELSZ)

print("J =", wav_op.J, "| L =", wav_op.L, "| KERNELSZ =", wav_op.KERNELSZ)
print("j_to_dg =", list(wav_op.j_to_dg))
print("mask_full_res =", wav_op.mask_full_res)

conv = wav_op.apply(data, 0)
assert conv.array.shape == (L, NPIX)
assert conv.conv_history == [0], conv.conv_history
assert conv.dg == 0 and conv.nside == NSIDE
assert torch.is_complex(conv.array)

fig = plt.figure(figsize=(16, 4))
for k in range(L):
    hp.mollview(
        conv.array[k].abs().cpu().numpy(),
        nest=True, hold=False, sub=(1, L, 1 + k),
        title=r"$|I*\psi|$ orientation %d" % k, cmap="viridis",
    )
plt.show()

J = 4 | L = 4 | KERNELSZ = 5
j_to_dg = [0, 1, 2, 3]
mask_full_res = None


### Second convolution and history

The `|I*psi_2| * psi_3` chain must produce `[..., L2, L3, Npix]` — that is what
`ST_Operator` expects for S3 and S4.

In [7]:
mod1 = conv.modulus()
conv2 = wav_op.apply(mod1, 0)

assert conv2.array.shape == (L, L, NPIX), conv2.array.shape
assert conv2.conv_history == [0, 0]
print("layer 1:", tuple(conv.array.shape), conv.conv_history)
print("layer 2:", tuple(conv2.array.shape), conv2.conv_history)

layer 1: (4, 12288) [0]
layer 2: (4, 4, 12288) [0, 0]


## 3. Downsampling

Signature aligned on the planar kernel: `downsample(data, dg_out, inplace=...,
replace_nan_value=...)`. The anti-aliased decimation is the one from
**`healpix-analyse`** (`HealPixDown`, `"smooth"` mode) applied one level at a
time; `smooth=False` falls back to the plain average of the 4 NESTED children.

In [8]:
cur = DataClass(heal_im)
print("dg=0: Npix =", cur.array.shape[-1], "| nside =", cur.nside)

fig = plt.figure(figsize=(16, 4))
hp.mollview(cur.array.cpu().numpy(), nest=True, hold=False, sub=(1, J, 1),
            title="dg = 0", cmap="plasma")
for dg in range(1, J):
    cur = wav_op.downsample(cur, dg, inplace=False)
    assert cur.dg == dg
    assert cur.nside == NSIDE // 2**dg
    assert cur.array.shape[-1] == 12 * cur.nside**2
    assert cur.cell_ids.shape[-1] == cur.array.shape[-1]
    hp.mollview(cur.array.cpu().numpy(), nest=True, hold=False, sub=(1, J, 1 + dg),
                title="dg = %d (nside %d)" % (dg, cur.nside), cmap="plasma")
plt.show()
print("downsampling: OK")

dg=0: Npix = 12288 | nside = 32


downsampling: OK


## 4. Statistics carried by the operator

This was the heart of the divergence: `mean`, `square_mean` and `cov` lived on
the data class with incompatible signatures. They now sit on the operator,
exactly as in `STL_2D_Kernel_Torch`.

In [9]:
x = DataClass(np.stack([heal_im, 2.0 * heal_im]))   # (2, Npix)

m = wav_op.mean(x)
s2 = wav_op.square_mean(x)
c = wav_op.cov(x, x)

assert m.shape == (2,) and s2.shape == (2,) and c.shape == (2,)
assert torch.allclose(m, x.array.mean(dim=-1))
assert torch.allclose(s2.real, (x.array**2).mean(dim=-1))
assert torch.allclose(c.real, s2.real)

print("mean        =", m.cpu().numpy())
print("square_mean =", s2.real.cpu().numpy())
print("cov(x, x)   =", c.real.cpu().numpy())

mean        = [3.45499092e-17 6.90998185e-17]
square_mean = [1. 4.]
cov(x, x)   = [1. 4.]


In [10]:
# standardize / unstandardize: exact round trip
std_data, mean_used, std_used = wav_op.standardize(x, mean_field=False, inplace=False)

# scalar comparisons, so the checks hold whatever device the tensors live on
mean_after = wav_op.mean(std_data).real
var_after = wav_op.cov(std_data, std_data).real
assert mean_after.abs().max().item() < 1e-10, mean_after
assert (var_after - 1.0).abs().max().item() < 1e-10, var_after

back = wav_op.unstandardize(std_data, mean_used, std_used, inplace=False)
err = (back.array - x.array).abs().max().item()
assert err < 1e-10, err

print("standardize: zero mean, unit variance — round trip within %.2e" % err)

standardize: zero mean, unit variance — round trip within 0.00e+00


## 5. Full `ST_Operator` chain: S1 to S4

The decisive test: the same data-type agnostic code as for the planar case,
applied to HEALPix data.

In [11]:
st_op = data.get_ST_op(J=J, L=L)
print("compute_PS =", st_op.compute_PS, "| n_bins =", st_op.n_bins)

st = st_op.apply(data, norm="vanilla")   # unnormalised: S2 would otherwise be 1 everywhere

expected = {
    "PS": (1, 1, 1, st_op.n_bins),
    "S1": (1, 1, 1, J, L),
    "S2": (1, 1, 1, J, L),
    "S3": (1, 1, 1, J, J, L, L),
    "S4": (1, 1, 1, J, J, J, L, L, L),
}
for name, shape in expected.items():
    got = tuple(getattr(st, name).shape)
    assert got == shape, (name, got, shape)
    print("%-3s %-28s  finite fraction = %.3f" % (
        name, str(got), torch.isfinite(getattr(st, name)).float().mean().item()))

print("mean =", st.mean.real.cpu().numpy(), "| var =", st.var.real.cpu().numpy())

compute_PS = True | n_bins = 20


PS  (1, 1, 1, 20)                 finite fraction = 1.000
S1  (1, 1, 1, 4, 4)               finite fraction = 1.000
S2  (1, 1, 1, 4, 4)               finite fraction = 1.000
S3  (1, 1, 1, 4, 4, 4, 4)         finite fraction = 0.625
S4  (1, 1, 1, 4, 4, 4, 4, 4, 4)   finite fraction = 0.312
mean = [[3.45499092e-17]] | var = [[1.]]


In [12]:
plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
for lidx in range(L):
    plt.plot(np.arange(J), st.S1[0, 0, 0, :, lidx].real.cpu().numpy(), marker="o",
             label="orientation %d" % lidx)
plt.yscale("log"); plt.xlabel("j"); plt.ylabel("S1"); plt.title("S1(j, theta)")
plt.legend(fontsize=8); plt.grid(alpha=.3)

plt.subplot(1, 3, 2)
for lidx in range(L):
    plt.plot(np.arange(J), st.S2[0, 0, 0, :, lidx].real.cpu().numpy(), marker="o")
plt.yscale("log"); plt.xlabel("j"); plt.ylabel("S2"); plt.title("S2(j, theta)")
plt.grid(alpha=.3)

plt.subplot(1, 3, 3)
s4 = st.S4.abs().cpu().numpy().flatten()
plt.plot(s4[np.isfinite(s4)], lw=.8)
plt.yscale("log"); plt.title("|S4| (computed coefficients)"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

### Batch, multiple channels and cross statistics

In [13]:
Nb, Nc = 2, 2
maps = np.stack([np.stack([heal_im, np.roll(heal_im, 137)]) for _ in range(Nb)])
maps = maps + 0.05 * np.random.randn(*maps.shape)
multi = DataClass(maps)                                   # (Nb, Nc, Npix)

cross = torch.triu(torch.ones((Nc, Nc), dtype=bool, device=multi.device))
st_multi = st_op.apply(multi, compute_cross_matrix=cross, norm="vanilla")

assert tuple(st_multi.S4.shape) == (Nb, Nc, Nc, J, J, J, L, L, L)
print("S1", tuple(st_multi.S1.shape))
print("S4", tuple(st_multi.S4.shape))
print("S1 finite fraction = %.2f (the channel sub-diagonal stays NaN, as in 2D)"
      % torch.isfinite(st_multi.S1).float().mean().item())

S1 (2, 2, 2, 4, 4)
S4 (2, 2, 2, 4, 4, 4, 4, 4, 4)
S1 finite fraction = 0.75 (the channel sub-diagonal stays NaN, as in 2D)


### `store_ref` / `load_ref` normalisation and standardisation

In [14]:
st_ref = st_op.apply(multi, compute_cross_matrix=cross, standardize=True, norm="store_ref")
assert st_ref.norm and st_ref.standardized

other = DataClass(maps + 0.3 * np.random.randn(*maps.shape))
st_load = st_op.apply(other, compute_cross_matrix=cross, norm="load_ref")
assert st_load.norm

st_iso = st_op.apply(multi, compute_cross_matrix=cross, norm="vanilla", iso=True)
print("store_ref / load_ref / iso: OK")
print("isotropised S3:", tuple(st_iso.S3.shape))

store_ref / load_ref / iso: OK
isotropised S3: (2, 2, 2, 4, 4, 4)


## 6. Angular power spectrum

`get_CS_op()` returns `CS_operator_Healpix_Torch`, the spherical counterpart of
`CS_operator_2D_Kernel_Torch`: same contract (`n_bins`, `bin_centers`,
`apply(...) -> [Nb, Nc, Nc, n_bins]`, `plot_cross_spectrum`), but the estimator
is C_ell instead of an FFT ring binning.

Two equivalent full-sky routes are available — `map2alm` once per channel then
every pair (`"alm"`, the default), or `HEALPixSHT.anafast` pair by pair
(`"anafast"`, the reference). On a partial sky the coefficients are band-filtered
with `alm2map` and the cross-covariance is taken over the observed pixels only.

The C_ell are averaged in logarithmically spaced multipole bins, weighted by
(2l+1) times a log-Gaussian window — the spherical transposition of the planar
`_build_log_gaussian_bin_masks`.

In [15]:
cs_op = data.get_CS_op()
print("lmax    =", cs_op.lmax)
print("n_bins  =", cs_op.n_bins)
print("bin centres (ell):", np.round(cs_op.bin_centers.cpu().numpy(), 1))

ps = cs_op.apply(data)
assert tuple(ps.shape) == (1, 1, 1, cs_op.n_bins)
assert torch.isfinite(ps).all()
print("PS shape:", tuple(ps.shape))

lmax    = 95
n_bins  = 20
bin centres (ell): [ 1.1  1.4  1.8  2.2  2.8  3.5  4.4  5.5  6.9  8.7 10.9 13.7 17.2 21.6
 27.2 34.1 42.8 53.8 67.5 84.8]
PS shape: (1, 1, 1, 20)


### The two full-sky routes agree, and match `anafast`

The internal cross-spectrum uses the very estimator `HEALPixSHT.anafast`
implements, so on an auto-spectrum the two must agree to machine precision.

In [16]:
sht = cs_op._get_sht()
# note: ST_Operator.apply reshapes data.array in place to (Nb, Nc, Npix),
# so the multipole axis is indexed explicitly here
alm = sht.map2alm(data.array, nest=True)

cl_internal = cs_op._cross_cl(alm, alm)
cl_anafast = sht.anafast(data.array, nest=True)
rel = (
    (cl_internal[..., 2:] - cl_anafast[..., 2:]).abs() / cl_anafast[..., 2:].abs()
).max().item()
print("internal C_ell vs HEALPixSHT.anafast : max relative %.2e" % rel)
assert rel < 1e-10

ps_alm = cs_op.apply(data, cross_spectrum_method="alm")
ps_anafast = cs_op.apply(data, cross_spectrum_method="anafast")
rel = ((ps_alm - ps_anafast).abs() / ps_anafast.abs()).max().real.item()
print("binned: 'alm' route vs 'anafast' route: max relative %.2e" % rel)
assert rel < 1e-10

internal C_ell vs HEALPixSHT.anafast : max relative 7.46e-16
binned: 'alm' route vs 'anafast' route: max relative 2.37e-16


### The band-filtered route reduces to the full-sky one

On a complete sky the pixel-space estimator built with `alm2map` must return
exactly the harmonic answer. This is what pins down its normalisation,
`4 pi / sum_l (2l+1) W_b(l)`.

In [17]:
ps_band = cs_op.apply(data, use_band_maps=True)
rel = ((ps_band - ps_alm).abs() / ps_alm.abs()).max().real.item()
print("band-filtered route vs harmonic route: max relative %.2e" % rel)
assert rel < 1e-10

band-filtered route vs harmonic route: max relative 3.53e-16


### Comparison with `healpy.anafast`

An independent check on the physics. The two spherical harmonic transforms use
different quadratures, so they part company as the multipole approaches the
pixel scale — which is why the deviation grows in the last bins.

In [18]:
cl_hp = hp.anafast(hp.reorder(heal_im, n2r=True), lmax=cs_op.lmax)
weights = cs_op.bin_weights.cpu().numpy()
cb_hp = (weights * cl_hp[None, :]).sum(1) / weights.sum(1)
cb_stl = ps[0, 0, 0].real.cpu().numpy()

ratio = cb_stl / cb_hp
low = cs_op.bin_centers.cpu().numpy() < cs_op.lmax / 2
print("ell < lmax/2 : max deviation %.2f %%" % (100 * np.abs(ratio[low] - 1).max()))
print("ell > lmax/2 : max deviation %.2f %%" % (100 * np.abs(ratio[~low] - 1).max()))

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
cs_op.plot_cross_spectrum(ps, label="STL / healpix-analyse", color="C0")
plt.plot(cs_op.bin_centers.cpu().numpy(), cb_hp, "--", color="C1", label="healpy.anafast")
plt.legend()
plt.subplot(1, 2, 2)
plt.semilogx(cs_op.bin_centers.cpu().numpy(), ratio, marker="o", color="C2")
plt.axhline(1.0, color="k", lw=.8)
plt.xlabel(r"multipole $\ell$"); plt.ylabel("STL / healpy"); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

ell < lmax/2 : max deviation 0.12 %
ell > lmax/2 : max deviation 7.16 %


### Cross-spectra, partial sky and gradient

In [19]:
# cross-channel spectra
ps_multi = cs_op.apply(multi, compute_cross_spectrum_matrix=cross)
assert tuple(ps_multi.shape) == (Nb, Nc, Nc, cs_op.n_bins)
print("cross spectra:", tuple(ps_multi.shape),
      "| finite fraction %.2f (only the requested pairs are filled)"
      % torch.isfinite(ps_multi).float().mean().item())

# partial sky: pseudo-C_ell over half the sphere
half = np.arange(NPIX // 2)
patch_half = DataClass(heal_im[half], nside=NSIDE, cell_ids=half)
ps_half = patch_half.get_CS_op().apply(patch_half)
recovered = (ps_half[0, 0, 0].real / ps[0, 0, 0].real).cpu().numpy()
print("half sky vs full sky: median ratio %.3f" % np.median(recovered))

# the spectrum is differentiable, so it can enter a synthesis loss
v = torch.tensor(heal_im, dtype=torch.float64, requires_grad=True)
cs_op.apply(DataClass(v)).real.sum().backward()
assert torch.isfinite(v.grad).all()
print("gradient through the spectrum: ||grad|| = %.3e, all finite" % v.grad.norm().item())

cross spectra: (2, 2, 2, 20) | finite fraction 0.75 (only the requested pairs are filled)


half sky vs full sky: median ratio 1.011
gradient through the spectrum: ||grad|| = 2.265e-02, all finite


## 7. Differentiability

Synthesis rests entirely on `backward()`: the gradient must flow back to the
input map through the spherical convolution.

In [20]:
u = torch.tensor(heal_im, dtype=torch.float64, requires_grad=True)
st_u = st_op.apply(DataClass(u), norm="vanilla")

loss = torch.nansum(st_u.S2.real) + st_u.var.real.sum()
loss.backward()

assert u.grad is not None and torch.isfinite(u.grad).all()
print("||grad|| = %.4e over %d pixels, all finite" % (u.grad.norm().item(), u.grad.numel()))

hp.mollview(u.grad.detach().cpu().numpy(), nest=True, cmap="coolwarm",
            title="d(loss)/d(pixel)")
plt.show()

||grad|| = 2.3516e-02 over 12288 pixels, all finite


## 8. Partial sky (`cell_ids`)

A NESTED subset of pixels: `pbc` switches to `False`, and `nside` has to be
given explicitly since it can no longer be read off `Npix`.

In [21]:
cell_ids = np.arange(NPIX // 4)              # one contiguous NESTED quarter of the sky
patch = DataClass(heal_im[cell_ids], nside=NSIDE, cell_ids=cell_ids)

assert patch.pbc is False, "partial sky -> pbc False"
assert patch.N0 == (NSIDE,) and patch.nside == NSIDE

wav_patch = patch.get_wavelet_op(J=3, L=L, kernel_size=KERNELSZ)
conv_patch = wav_patch.apply(patch, 0)
down_patch = wav_patch.downsample(conv_patch.modulus(), 1, inplace=False)

print("patch: Npix =", patch.array.shape[-1], "| pbc =", patch.pbc)
print("conv :", tuple(conv_patch.array.shape))
print("down :", tuple(down_patch.array.shape), "| nside =", down_patch.nside)

st_patch = patch.get_ST_op(J=3, L=L).apply(patch, norm="vanilla")
print("S2 on the patch:", tuple(st_patch.S2.shape),
      "| finite at %.2f" % torch.isfinite(st_patch.S2).float().mean().item())

patch: Npix = 3072 | pbc = False
conv : (4, 3072)
down : (4, 768) | nside = 16


S2 on the patch: (1, 1, 1, 3, 4) | finite at 1.00


## 9. NaNs: what already works, what is missing

Without a mask, NaNs propagate — the same behaviour as the planar kernel without
`mask_full_res`. The `nan_aware_stats=True` option makes the **reductions**
(`mean`, `cov`, ...) and the downsampling ignore NaNs.

Note that the **convolutions** are still unprotected, so a NaN pixel
contaminates its neighbourhood at every layer. The complete fix is step 5 of the
plan (per-layer masks obtained by eroding the stencil support).

In [22]:
nan_map = heal_im.copy()
nan_map[(np.random.rand(200) * NPIX).astype(int)] = np.nan
d_nan = DataClass(nan_map)

# the angular power spectrum is undefined on NaNs (it would need cell_ids
# instead), so this part of the chain runs with compute_PS=False
st_plain = d_nan.get_ST_op(J=J, L=L, compute_PS=False).apply(
    DataClass(nan_map), norm="vanilla"
)
st_aware = d_nan.get_ST_op(
    J=J, L=L, compute_PS=False, wavelet_op_kwargs={"nan_aware_stats": True}
).apply(DataClass(nan_map), norm="vanilla")

print("unprotected     : S2 finite at %.2f (NaNs contaminate everything)"
      % torch.isfinite(st_plain.S2).float().mean().item())
print("nan_aware_stats : S2 finite at %.2f, S4 finite at %.2f"
      % (torch.isfinite(st_aware.S2).float().mean().item(),
         torch.isfinite(st_aware.S4).float().mean().item()))

# The coefficients become finite again, but not exact: the convolutions spread
# every NaN over its neighbourhood. The residual bias is measured against the
# clean map.
print("(%.1f %% of the pixels masked)" % (100 * np.isnan(nan_map).mean()))
ref, got = st.S2.real, st_aware.S2.real
dev = ((got - ref).abs() / ref.abs()).flatten()
print()
print("residual bias on S2: median %.2e, max %.2e"
      % (dev.median().item(), dev.max().item()))
print("-> this is what step 5 (per-layer masks) has to remove.")

unprotected     : S2 finite at 0.00 (NaNs contaminate everything)
nan_aware_stats : S2 finite at 1.00, S4 finite at 0.31
(1.6 % of the pixels masked)

residual bias on S2: median 4.31e-03, max 4.33e-02
-> this is what step 5 (per-layer masks) has to remove.


## 10. Interface comparison with the planar kernel

A programmatic check: which public members of `STL_2D_Kernel_Torch` and of its
operator are still missing on the HEALPix side?

In [23]:
from STL_main.STL_2D_Kernel_Torch import (
    STL_2D_Kernel_Torch,
    WaveletOperator2Dkernel_torch,
)

# instances are inspected, since several members (array, device, dtype,
# mask_full_res, ...) only exist once the object is built
planar = STL_2D_Kernel_Torch(array=np.random.rand(64, 64), pbc=True)
planar_op = planar.get_wavelet_op(J=3, L=L)


def api(obj):
    return {n for n in dir(obj) if not n.startswith("__")}


REQUIRED_OPERATOR = {
    "J", "L", "WType", "device", "dtype", "mask_full_res", "j_to_dg",
    "apply", "mean", "square_mean", "cov", "standardize", "unstandardize",
    "downsample", "_compute_and_store_cross_cov", "_find_mask",
}
REQUIRED_DATA = {
    "DT", "N0", "dg", "pbc", "conv_history", "array", "device", "dtype",
    "copy", "modulus", "divide", "get_wavelet_op", "get_ST_op", "get_CS_op",
}

missing_data = REQUIRED_DATA - api(data)
missing_op = REQUIRED_OPERATOR - api(wav_op)

print("required members missing (data class):", missing_data or "none")
print("required members missing (operator)  :", missing_op or "none")
assert not missing_data and not missing_op

gap_data = sorted(api(planar) - api(data))
gap_op = sorted(api(planar_op) - api(wav_op))
print()
print("present in 2D but not in HEALPix (expected: planar specifics):")
print("  data    :", gap_data or "none")
print("  operator:", gap_op or "none")

required members missing (data class): none
required members missing (operator)  : none

present in 2D but not in HEALPix (expected: planar specifics):
  data    : none
  operator: ['_build_bump_steerable_wavelet_kernel', '_build_morlet_wavelet_kernel', '_build_reweighting_maps_and_scattering_layer_masks', '_build_wavelet_kernel_from_ifft_crop', '_conv2d_circular', '_crop', '_downsample_tensor', '_get_crop_border_size_fully_flexible', '_get_crop_border_size_largest_scale_layer_flexible', '_get_crop_border_size_largest_scale_second_layer', '_get_padding_mode', '_get_smooth_kernel', '_layer1_mask', '_layer2_mask', '_reweighting_maps_smooth', '_reweighting_maps_wav', '_semicomplex_conv2d_circular', '_wav_kernel']


In [24]:
# what is unsupported must fail explicitly, not silently
try:
    data.get_wavelet_op(J=J, mask_full_res=DataClass(np.isnan(nan_map).astype(float)))
except NotImplementedError as exc:
    print("mask_full_res ->", str(exc)[:110], "...")

try:
    cs_op.apply(DataClass(nan_map))
except ValueError as exc:
    print("spectrum on NaNs ->", str(exc)[:110], "...")

mask_full_res -> mask_full_res is not supported yet by the HEALPix kernel. The spherical counterpart of the planar layer masks  ...
spectrum on NaNs -> Data array contains NaN values; the angular power spectrum cannot be computed on them. Mask them out through c ...


## Summary

Certified by this notebook:

| | status |
|---|---|
| data class conforming to `Base_DataClass` | OK |
| `divide`, `get_ST_op`, `copy`, `__getitem__`, `modulus` | OK |
| wavelet convolution (`HealPixConv`) + `conv_history` | OK |
| anti-aliased downsampling (`HealPixDown`) | OK |
| `mean` / `square_mean` / `cov` / `standardize` on the operator | OK |
| S1–S4 chain, batch, channels, cross statistics | OK |
| `store_ref` / `load_ref` normalisation, `iso` | OK |
| differentiability (synthesis) | OK |
| partial sky through `cell_ids` | OK |
| angular power spectrum (`HEALPixSHT`) | OK |
| full mask / NaN handling (`mask_full_res`) | pending, step 5 |
| pseudo-C_ell mode-coupling deconvolution | pending |
| generalised `Synthesis` | pending, step 7 |

Still to do as well: align the radial profile and the angle convention of the
kernel with those of the planar kernel, so that the coefficients of the two data
types are directly comparable (step 8).